In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
train_data = pd.read_csv('../data/new_train_data_39k.csv')
train_data.info()

In [ ]:
train_data['joke'] = train_data['setup'] + ' ' + train_data['punchline']
train_data = train_data.dropna()

train_data['final_score'].describe()

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

# Calculate median for binary classification
# train_data = train_data.sample(100000, random_state=42)
median_score = train_data['final_score'].median()
# train_data['binary_label'] = (train_data['final_score'] > median_score).astype(int)
train_data['binary_label'] = (train_data['final_score'] > 0.2249769757430212).astype(int)

# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_texts(texts, max_len=128):
    return tokenizer(
        text=texts.tolist(),
        add_special_tokens=True,
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )

# Create DataLoaders
def create_dataloader(inputs, masks, labels, batch_size=64):
    dataset = TensorDataset(inputs, masks, labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Define Model
class BertClassifier(nn.Module):
    def __init__(self, dropout=0.1):
        super(BertClassifier, self).__init__()
        self.bert = BertForSequenceClassification.from_pretrained(
            'bert-base-uncased',
            num_labels=1,
            problem_type="binary_classification"
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        return outputs.logits

# Training Function
def train_model(model, train_dataloader, val_dataloader, epochs=10):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    criterion = nn.BCEWithLogitsLoss()

    best_accuracy = 0
    best_model_state = None

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        # Training phase
        model.train()
        total_loss = 0
        train_preds, train_labels = [], []
        progress_bar = tqdm(train_dataloader, desc='Training', unit='batch')

        for batch in progress_bar:
            inputs, masks, labels = [b.to(device) for b in batch]
            
            optimizer.zero_grad()
            outputs = model(inputs, masks).squeeze()
            loss = criterion(outputs, labels.float())
            
            loss.backward()
            optimizer.step()
            
            # Calculate training predictions
            preds = (torch.sigmoid(outputs) > 0.5).int()
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
        
        # Calculate training metrics
        avg_train_loss = total_loss / len(train_dataloader)
        train_accuracy = accuracy_score(train_labels, train_preds)
        print(f"Average Training Loss: {avg_train_loss:.4f}")
        print(f"Training Accuracy: {train_accuracy:.4f}")
        
        # Print training classification report
        train_report = classification_report(train_labels, train_preds, target_names=['Class 0', 'Class 1'])
        print(f"\nTraining Classification Report:\n{train_report}")
        
        # Validation phase
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            val_progress_bar = tqdm(val_dataloader, desc='Validating', unit='batch')
            for batch in val_progress_bar:
                inputs, masks, labels = [b.to(device) for b in batch]
                outputs = torch.sigmoid(model(inputs, masks).squeeze())
                preds = (outputs > 0.5).int()
                
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        val_accuracy = accuracy_score(val_labels, val_preds)
        print(f"Validation Accuracy: {val_accuracy:.4f}")
        
        # Print validation classification report
        val_report = classification_report(val_labels, val_preds, target_names=['Class 0', 'Class 1'])
        print(f"\nValidation Classification Report:\n{val_report}")
        
        # Save the best model based on validation accuracy
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_model_state = model.state_dict()
            print("Best model updated.")
    
    # Save the best model to disk
    if best_model_state is not None:
        torch.save(best_model_state, 'best_bert_model2_filtered2.pth')
        print(f"Best model saved with validation accuracy: {best_accuracy:.4f}")

# Prepare Data
encoded = tokenize_texts(train_data['joke'])
input_ids = encoded['input_ids']
attention_mask = encoded['attention_mask']
labels = torch.tensor(train_data['binary_label'].values)

# Split into training and validation sets
X_train, X_val, mask_train, mask_val, y_train, y_val = train_test_split(
    input_ids, attention_mask, labels, test_size=0.25, random_state=42
)

# Create DataLoaders
train_dataloader = create_dataloader(X_train, mask_train, y_train)
val_dataloader = create_dataloader(X_val, mask_val, y_val)

# Train the model
model = BertClassifier()
train_model(model, train_dataloader, val_dataloader, epochs=5)